# OASIS-1: SynthSeg segmentation
Settings: **Internet ON**, **Accelerator: GPU T4 x2** (uses both GPUs in parallel; one GPU also works). Without a GPU it falls back to CPU (~6 min per scan, too slow for all 436). Attach the private dataset with the `.nii.gz` images. The SynthSeg 2.0 weights are pulled from the FreeSurfer 7.4.1 download automatically (~15-30 min, once).

Run with **Save Version → Save & Run All**. It runs up to ~11 h, saves results every 20 scans, and stops cleanly before Kaggle's time limit.
If it does not finish, add this notebook's output as an input dataset and run it again: finished scans are skipped.

In [ ]:
import glob, os, shutil, subprocess, time
T_START = time.time()
TIME_LIMIT_H = 11.0
OUT = "/kaggle/working/synthseg"
os.makedirs(OUT, exist_ok=True)

# walk /kaggle/input following links; accept .nii.gz or .nii (Kaggle may unpack .gz on upload)
found = []
for root, dirs, files in os.walk("/kaggle/input", followlinks=True):
    found += [os.path.join(root, f) for f in files]
images = sorted(p for p in found if os.path.basename(p).startswith("OAS1_")
                and p.endswith((".nii.gz", ".nii")) and "synthseg" not in p)
weights = {os.path.basename(p): p for p in found if p.endswith(".h5")}
print("files under /kaggle/input:", len(found), "e.g.", found[:3])
need = ["synthseg_2.0.h5", "synthseg_parc_2.0.h5", "synthseg_qc_2.0.h5"]
print("images found:", len(images))
print("weights found:", sorted(weights))
assert images, "No OAS1_*.nii.gz images under /kaggle/input"

# resume: copy segmentations from an earlier run of this notebook, if attached
# matched by file name, so it works whether the old output is attached as notebook output
# or re-uploaded as a dataset (Kaggle may flatten folders or unpack .gz)
prev = [f for f in found if "_synthseg.nii" in os.path.basename(f)
        or os.path.basename(f).startswith(("vol_OAS1", "qc_OAS1"))]
print("previous results found:", len(prev))
for p in prev:
    dst = os.path.join(OUT, os.path.basename(p))
    if not os.path.exists(dst):
        shutil.copy(p, dst)

In [ ]:
# Python 3.8 + TensorFlow 2.2 in /tmp, so it is not saved as notebook output.
# TF 2.2 needs CUDA 10.1 + cuDNN 7.6, so the env brings its own copies (Kaggle's are newer).
USE_GPU = shutil.which("nvidia-smi") is not None
print("GPU present:", USE_GPU)
!curl -Ls https://micro.mamba.pm/api/micromamba/linux-64/latest | tar -xj -C /tmp bin/micromamba
if USE_GPU:
    !/tmp/bin/micromamba create -y -q -p /tmp/ss38 -c conda-forge python=3.8 cudatoolkit=10.1 cudnn=7.6.5 > /dev/null
    !/tmp/ss38/bin/pip install -q tensorflow-gpu==2.2.0 keras==2.3.1 protobuf==3.20.3 numpy==1.23.5 nibabel==5.0.1 matplotlib==3.6.2
else:
    !/tmp/bin/micromamba create -y -q -p /tmp/ss38 -c conda-forge python=3.8 > /dev/null
    !/tmp/ss38/bin/pip install -q tensorflow-cpu==2.2.0 keras==2.3.1 protobuf==3.20.3 numpy==1.23.5 nibabel==5.0.1 matplotlib==3.6.2
ENV = dict(os.environ, LD_LIBRARY_PATH="/tmp/ss38/lib:" + os.environ.get("LD_LIBRARY_PATH", ""))
!git clone -q https://github.com/BBillot/SynthSeg /tmp/SynthSeg

# weights: from an attached earlier output if present, else stream them out of the FreeSurfer 7.4.1 tarball
FS_URL = "https://surfer.nmr.mgh.harvard.edu/pub/dist/freesurfer/7.4.1/freesurfer-linux-ubuntu22_amd64-7.4.1.tar.gz"
if not all(w in weights for w in need):
    pattern = " ".join("'*/models/%s'" % w for w in need)
    !curl -sL {FS_URL} | tar -xz -C /tmp --wildcards {pattern}
    for w in need:
        hits = glob.glob("/tmp/**/models/" + w, recursive=True)
        assert hits, "Could not find %s in the FreeSurfer download" % w
        weights[w] = hits[0]
os.makedirs("/kaggle/working/weights", exist_ok=True)
for w in need:
    shutil.copy(weights[w], "/tmp/SynthSeg/models/")
    shutil.copy(weights[w], "/kaggle/working/weights/")   # saved, so a rerun skips the big download
print("weights ready:", need)
chk = subprocess.run(["/tmp/ss38/bin/python", "-c",
    "import tensorflow as tf, keras; print('TF', tf.__version__, 'Keras', keras.__version__, 'GPUs', tf.config.list_physical_devices('GPU'))"],
    capture_output=True, text=True, env=ENV)
print(chk.stdout, chk.stderr[-1500:])
if USE_GPU and "GPUs []" in chk.stdout:
    print("WARNING: TensorFlow cannot see the GPU, falling back to CPU (slow).")
    USE_GPU = False

In [ ]:
import threading
from concurrent.futures import ThreadPoolExecutor

PY = "/tmp/ss38/bin/python"
SCRIPT = "/tmp/SynthSeg/scripts/commands/SynthSeg_predict.py"
CHUNK = 20
N_GPU = len(subprocess.run(["nvidia-smi", "-L"], capture_output=True, text=True).stdout.strip().splitlines()) if USE_GPU else 0
WORKERS = max(1, N_GPU)
print("GPUs:", N_GPU, "-> parallel workers:", WORKERS)
stats = {"n": 0, "t0": None}
FAILED = {}   # scan -> error text
lock = threading.Lock()

def seg_done(p):
    return bool(glob.glob(os.path.join(OUT, os.path.basename(p).split(".nii")[0] + "_synthseg.nii*")))

def run_chunk(chunk, worker=0):
    cdir = f"/tmp/chunk{worker}"
    shutil.rmtree(cdir, ignore_errors=True)
    os.makedirs(cdir)
    for p in chunk:
        os.symlink(p, os.path.join(cdir, os.path.basename(p)))
    tag = os.path.basename(chunk[0]).split(".nii")[0]
    env = dict(ENV)
    if USE_GPU:
        env["CUDA_VISIBLE_DEVICES"] = str(worker)
    cmd = [PY, SCRIPT, "--i", cdir, "--o", OUT, "--parc",
           "--vol", f"{OUT}/vol_{tag}.csv", "--qc", f"{OUT}/qc_{tag}.csv",
           "--threads", str(max(1, os.cpu_count() // WORKERS))] + ([] if USE_GPU else ["--cpu"])
    r = subprocess.run(cmd, capture_output=True, text=True, env=env)
    if r.returncode != 0:
        if len(chunk) > 1:
            # retry one scan at a time, so one bad scan cannot block the others
            print(f"[worker {worker}] chunk starting at {tag} failed, retrying its scans one by one", flush=True)
            for p in chunk:
                if not seg_done(p):
                    run_chunk([p], worker)
            return
        with lock:
            FAILED[tag] = (r.stdout[-1500:] + "\n" + r.stderr[-1500:]).strip()
            print(f"[worker {worker}] FAILED {tag}:\n{FAILED[tag]}", flush=True)
        return
    with lock:
        stats["n"] += len(chunk)
        left = sum(not seg_done(p) for p in images)
        per = (time.time() - stats["t0"]) / stats["n"]   # wall-clock seconds per scan, all workers together
        print(f"[worker {worker}] done {len(images) - left}/{len(images)}, {per:.0f} s per scan overall, about {per * left / 3600:.1f} h left", flush=True)

def run_shard(shard, worker):
    for k in range(0, len(shard), CHUNK):
        if (time.time() - T_START) / 3600 > TIME_LIMIT_H:
            print(f"[worker {worker}] stopping before Kaggle's time limit. Rerun with this output attached.")
            return
        run_chunk(shard[k:k + CHUNK], worker)

# quick check on 5 scans first (one worker): if this fails, nothing else runs
todo = [p for p in images if not seg_done(p)]
print(len(images) - len(todo), "already done,", len(todo), "to run")
stats["t0"] = time.time()
if todo:
    run_chunk(todo[:5], 0)

In [ ]:
# the rest: split across the GPUs, chunks of 20, stopping before Kaggle's time limit
todo = [p for p in images if not seg_done(p)]
shards = [todo[w::WORKERS] for w in range(WORKERS)]
stats["n"], stats["t0"] = 0, time.time()
with ThreadPoolExecutor(WORKERS) as ex:
    for f in [ex.submit(run_shard, sh, w) for w, sh in enumerate(shards)]:
        f.result()   # re-raises any worker error

In [ ]:
# merge the per-chunk CSVs
import pandas as pd
for kind in ("vol", "qc"):
    parts = sorted(glob.glob(f"{OUT}/{kind}_*.csv"))
    if parts:
        df = pd.concat([pd.read_csv(p) for p in parts]).drop_duplicates(subset=None)
        df.to_csv(f"/kaggle/working/{kind}_all.csv", index=False)
        print(kind, "rows:", len(df))
if FAILED:
    with open("/kaggle/working/failed_scans.txt", "w") as f:
        for k, v in FAILED.items():
            f.write(f"=== {k}\n{v}\n")
    print("FAILED scans (details in failed_scans.txt):", sorted(FAILED))
n = len(glob.glob(f"{OUT}/*_synthseg.nii*"))
print(f"segmentations: {n} of {len(images)}", "- COMPLETE" if n == len(images) else "- NOT complete, rerun")